# Neural Network Test Run (4) — Plain PyTorch version
Same model and data as the Lightning notebook, but with an explicit training loop (no `lightning` dependency).

In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
from torch.optim import Adam 
from torch.utils.data import TensorDataset, DataLoader 

import pandas as pd 
from sklearn.model_selection import train_test_split

## Load and prepare the data

In [2]:
df = pd.read_table('./iris.txt', sep=",", header=None)
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "class"]

In [3]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [4]:
df.shape

(150, 5)

In [5]:
df['class'].nunique()

3

In [6]:
df['class'].unique()

<ArrowStringArray>
['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
Length: 3, dtype: str

In [7]:
input_values = df[['petal_width', 'sepal_width']]
input_values.head()

,petal_width,sepal_width
0,0.2,3.5
1,0.2,3.0
2,0.2,3.2
3,0.2,3.1
4,0.2,3.6


In [9]:
label_values = df['class']
label_values.head()

0    Iris-setosa
1    Iris-setosa
2    Iris-setosa
3    Iris-setosa
4    Iris-setosa
Name: class, dtype: str

In [10]:
classes_as_numbers = label_values.factorize()[0]
classes_as_numbers

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [11]:
input_train, input_test, label_train, label_test = train_test_split(input_values, 
                                                                    classes_as_numbers, 
                                                                    test_size=0.25, 
                                                                    stratify=classes_as_numbers, 
                                                                    random_state=42)

In [12]:
input_train.shape

(112, 2)

In [14]:
label_train.shape

(112,)

In [13]:
input_test.shape

(38, 2)

In [15]:
label_test.shape

(38,)

In [17]:
one_hot_label_train = F.one_hot(torch.tensor(label_train)).type(torch.float32)


In [18]:
label_train

array([2, 2, 1, 1, 1, 2, 0, 2, 0, 2, 0, 2, 1, 0, 0, 1, 2, 0, 0, 1, 1, 1,
       0, 1, 2, 0, 2, 1, 2, 0, 0, 1, 0, 2, 0, 0, 1, 0, 1, 0, 0, 1, 2, 2,
       0, 2, 1, 0, 2, 0, 2, 2, 0, 1, 2, 2, 1, 1, 0, 1, 1, 2, 1, 2, 0, 1,
       0, 2, 1, 2, 1, 2, 2, 0, 2, 1, 0, 2, 0, 2, 1, 1, 0, 2, 2, 0, 0, 2,
       2, 1, 2, 0, 2, 1, 2, 2, 0, 1, 1, 1, 1, 1, 0, 2, 1, 1, 0, 0, 0, 0,
       1, 0])

In [20]:
one_hot_label_train[:10]

tensor([[0., 0., 1.],
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.]])

In [21]:
# Min-max normalize using TRAIN statistics only
max_vals_in_input_train = input_train.max()
min_vals_in_input_train = input_train.min()

# Normalize with max and min.
input_train = (input_train - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_train.head()

,petal_width,sepal_width
130,0.750000,0.333333
122,0.791667,0.333333
81,0.375000,0.166667
71,0.500000,0.333333
89,0.500000,0.208333


In [22]:
# Normalize test
input_test = (input_test - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_test.head()

,petal_width,sepal_width
42,0.041667,0.500000
56,0.625000,0.541667
99,0.500000,0.333333
53,0.500000,0.125000
38,0.041667,0.416667


In [23]:
# DataLoader lets us load data in batches, and can be used to train NN.
# Convert input_train into TENSORS
input_train_tensors = torch.tensor(input_train.values).type(torch.float32)
input_train_tensors[:5]

tensor([[0.7500, 0.3333],
        [0.7917, 0.3333],
        [0.3750, 0.1667],
        [0.5000, 0.3333],
        [0.5000, 0.2083]])

In [24]:
# Convert input_test into TENSORS
input_test_tensors = torch.tensor(input_test.values).type(torch.float32)
input_test_tensors[:5]

tensor([[0.0417, 0.5000],
        [0.6250, 0.5417],
        [0.5000, 0.3333],
        [0.5000, 0.1250],
        [0.0417, 0.4167]])

In [27]:
train_dataset = TensorDataset(input_train_tensors, one_hot_label_train)
train_dataloader = DataLoader(train_dataset) 

## Model
Plain `nn.Module`: `__init__` and `forward` only. The loss, optimizer, and training step now live outside the class.

In [28]:
class MultipleInsOuts(nn.Module):
  
  def __init__(self):
    super().__init__()
    
    self.input_to_hidden = nn.Linear(in_features=2, out_features=2, bias=True)
    
    self.hidden_to_output = nn.Linear(in_features=2, out_features=3, bias=True)
    
  
  def forward(self, x):
    hidden = self.input_to_hidden(x)
    return self.hidden_to_output(torch.relu(hidden)) # raw logits; no softmax
  

# replaces L.seed_everything(42)
torch.manual_seed(42)
model = MultipleInsOuts()
loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)

## Training loop and evaluation
This replaces `training_step`, `configure_optimizers`, and `Trainer.fit`.

In [29]:
def train(model, dataloader, loss_fn, optimzier, epochs):
  model.train()
  
  for epoch in range(epochs):
    total_loss = 0.0
    
    for inputs, labels in dataloader:
      # Clears old gradients.
      optimizer.zero_grad()
      
      # Forward Pass
      outputs = model(inputs)
      
      # Compute loss
      loss = loss_fn(outputs, labels)
      
      # Backpropagation
      loss.backward()
      
      # Update weights
      optimizer.step()
      total_loss += loss.item()
      
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}: avg loss = {total_loss / len(dataloader):.4f}")

def accuracy(model, inputs, labels):
    model.eval()
    with torch.no_grad():
        predicted = torch.argmax(model(inputs), dim=1)
    return (predicted == torch.tensor(labels)).float().mean().item()

In [30]:
train(model, train_dataloader, loss_fn, optimizer, epochs=10)
print("Test accuracy after 10 epochs:", accuracy(model, input_test_tensors, label_test))

Epoch 1: avg loss = 1.0385
Epoch 10: avg loss = 0.7897
Test accuracy after 10 epochs: 0.6578947305679321


* Continue training for 90 more epochs (100 total). The same `model` and `optimizer` objects keep their state, so no checkpoint is needed.

In [31]:
train(model, train_dataloader, loss_fn, optimizer, epochs=90)
print("Test accuracy after 100 epochs:", accuracy(model, input_test_tensors, label_test))

Epoch 1: avg loss = 0.7595
Epoch 10: avg loss = 0.5544
Epoch 20: avg loss = 0.4438
Epoch 30: avg loss = 0.3676
Epoch 40: avg loss = 0.3104
Epoch 50: avg loss = 0.2660
Epoch 60: avg loss = 0.2315
Epoch 70: avg loss = 0.2045
Epoch 80: avg loss = 0.1831
Epoch 90: avg loss = 0.1662
Test accuracy after 100 epochs: 0.9473684430122375


In [32]:
for name, param in model.named_parameters():
    print(name, torch.round(param.data, decimals=2))

input_to_hidden.weight tensor([[ 3.5500,  0.2500],
        [-1.9600,  1.4800]])
input_to_hidden.bias tensor([-0.3400,  1.2400])
hidden_to_output.weight tensor([[-4.2000,  3.1500],
        [ 0.1600,  0.3300],
        [ 1.8700, -3.2900]])
hidden_to_output.bias tensor([ 0.4600,  0.9600, -0.6100])


## Predict a single flower

In [34]:
normalized_values = ([0.2, 3.0] - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_tensor = torch.tensor(normalized_values.values, dtype=torch.float32)

model.eval()
with torch.no_grad():
    print(torch.argmax(model(input_tensor)))

tensor(0)
